# 11 — Stacking Modeli: General Panel (In-Silico Skorlar OLMADAN)

**Amac**: LightGBM, XGBoost, DNN ve SVM baz modelleri OOF (Out-of-Fold) stacking ile egitilir.

**Mimari**:
1. `X_cv` uzerinde 5-fold OOF: her fold'da egitilip, o fold'un validation kumesinde
   `[p_benign, p_patojenik]` skorlari uretilir → temiz meta-feature matrisi (1×8 vektor)
2. Final baz modeller tum `X_cv` ile yeniden egitilir → hold-out uzerinde skor uretir
3. Meta-model (LightGBM ve NN), OOF skorlari uzerinde egitilir; hold-out uzerinde degerlendirilir

**Panel**: Yalnizca General
**Train/Test bolunmesi**: notebook 07 ile birebir ayni (`TEST_SIZE=0.20`, `stratify=y`, `SEED=42`)
**Ozellikler**: Tum biyolojik feature'lar (in-silico YOK, `v3_no_insil` pipeline)


In [12]:
# Cell 1: Imports & Config
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, DATA_PATH, REPORTS_DIR
from src.features import prepare_data_v3_no_insil
from src.models import (
    grid_search_lightgbm, grid_search_xgboost,
    grid_search_dnn_fast, grid_search_svm
)
from src.metrics import optimize_threshold, compute_all_metrics
from src.utils import prepare_for_xgb, prepare_for_nn

RESULTS_STACK_DIR = os.path.join(PROJECT_ROOT, 'results', 'v3_stacking_general')
os.makedirs(RESULTS_STACK_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

PANEL = 'General'
BASE_MODELS = ['lightgbm', 'xgboost', 'dnn', 'svm']

print(f'Proje koku: {PROJECT_ROOT}')
print(f'Sonuc dizini: {RESULTS_STACK_DIR}')


Proje koku: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
Sonuc dizini: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v3_stacking_general


In [13]:
# Cell 2: Veri Yukleme & Feature Engineering
df_raw = pd.read_csv(DATA_PATH)
print(f'Ham veri: {df_raw.shape}')

sig = df_raw['clinvar__sig'].str.lower().str.strip()
target_map = {
    'benign': 0, 'likely benign': 0,
    'pathogenic': 1, 'likely pathogenic': 1,
}
df_raw['target'] = sig.map(target_map)
df_raw = df_raw.dropna(subset=['target'])
df_raw['target'] = df_raw['target'].astype(int)
print(f'Target dagilimi:\n{df_raw["target"].value_counts()}')

panel_series = df_raw['Panel'].copy()

df_v3 = prepare_data_v3_no_insil(df_raw)
df_v3['Panel'] = panel_series.values

print(f'FE sonrasi: {df_v3.shape}')
print(f'Panel dagilimi:\n{df_v3["Panel"].value_counts()}')

Ham veri: (4287, 119)
Target dagilimi:
target
1    2951
0    1336
Name: count, dtype: int64
  71 in-silico sutunu droplandi
  OHE (ayri alfabe): 48 feature
  K-mer DNA_11mer_Ref: 16 2-mer feature
  K-mer DNA_11mer_Alt: 16 2-mer feature
  K-mer Prot_11mer_Ref: 430 2-mer feature
  K-mer Prot_11mer_Alt: 434 2-mer feature
  24 gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 14 sutun dusuruldu
[V3-NoInSil] Veri boyutu: 4287 x 954
FE sonrasi: (4287, 954)
Panel dagilimi:
Panel
General              3156
Hereditary_Cancer     715
PAH                   324
CFTR                   92
Name: count, dtype: int64


In [14]:
# Cell 3: General Panel Bolunmesi (07 ile birebir ayni mantik)
df_panel = df_v3[df_v3['Panel'] == PANEL].copy()
df_panel.drop(columns=['Panel'], inplace=True)

X = df_panel.drop(columns=['target'])
y = df_panel['target']
print(f'General panel: {len(y)} satir  (pos={y.sum()}, neg={(y==0).sum()})')

# 07 ile ayni bolunme: %80 CV, %20 hold-out
X_cv, X_holdout, y_cv, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f'CV seti : {len(y_cv)} satir  (baz model OOF egitimi + meta model egitimi)')
print(f'Hold-out: {len(y_holdout)} satir  (final test, hicbir modele gosterilmedi)')

# OOF icin fold sayisi
N_FOLDS = 5
print(f'OOF fold sayisi: {N_FOLDS}')

General panel: 3156 satir  (pos=2182.0, neg=974)
CV seti : 2524 satir  (baz model OOF egitimi + meta model egitimi)
Hold-out: 632 satir  (final test, hicbir modele gosterilmedi)
OOF fold sayisi: 5


In [15]:
# Cell 4: Baz Modelleri OOF ile Egit & Ham Skor Uret
#
# Her baz model icin:
#   1. N_FOLDS-fold OOF: X_cv uzerinde her fold'da egit, val fold uzerinde skor uret
#      -> oof_probs[model] shape: (len(X_cv), 2)  -- meta-model egitimi icin
#   2. Final model: tum X_cv uzerinde en iyi hyperparametre ile egit
#      -> holdout_probs[model] shape: (len(X_holdout), 2) -- final degerlendirme icin
#
# Tum sabit parametreler src/models.py'den import edilir -> 07 ile tam ayni model

import lightgbm as lgb
import xgboost as xgb_lib
import torch.optim as optim
from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

# src/models.py'deki sabit parametreleri import et (07 ile senkronize kalir)
from src.models import (
    DeepMLP,
    LGBM_GRID, LGBM_FIXED,
    XGB_GRID,  XGB_FIXED,
    DNN_GRID_FAST, DNN_FIXED_FAST,
    SVM_GRID,  SVM_FIXED,
    _grid_combos,
)
from sklearn.preprocessing import StandardScaler as SKStandardScaler

cat_features = X_cv.select_dtypes(include=['category']).columns.tolist()

base_results   = {}
base_train_log = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

X_cv_reset = X_cv.reset_index(drop=True)
y_cv_reset = y_cv.reset_index(drop=True)

for mt_name in BASE_MODELS:
    print(f"\n{'='*60}")
    print(f"BAZ MODEL (OOF): {mt_name.upper()}")
    print(f"{'='*60}")
    t0 = time.time()

    grid = list(_grid_combos(
        LGBM_GRID if mt_name == 'lightgbm' else
        XGB_GRID  if mt_name == 'xgboost'  else
        DNN_GRID_FAST if mt_name == 'dnn'  else
        SVM_GRID
    ))
    print(f'  Grid: {len(grid)} kombinasyon')

    oof_probs   = np.zeros((len(X_cv_reset), 2))
    combo_votes = {}

    # XGBoost ve SVM icin X_cv'yi bir kez encode et
    if mt_name in ('xgboost', 'svm'):
        X_cv_enc_full, X_ho_enc_full, _le = prepare_for_xgb(X_cv_reset, X_holdout, cat_features)
        if mt_name == 'svm':
            X_cv_enc_full = X_cv_enc_full.fillna(0).astype(float)
            X_ho_enc_full = X_ho_enc_full.fillna(0).astype(float)
    # DNN icin X_cv'yi bir kez scale et (scaler X_cv ile fit edilmeli)
    if mt_name == 'dnn':
        nn_data_full = prepare_for_nn(X_cv_reset, X_holdout, y_cv_reset, y_holdout, cat_features)
        X_cv_t_full, X_ho_t_full, y_cv_t_full, _, input_dim_full, pos_weight_full, _ = nn_data_full

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_cv_reset, y_cv_reset)):
        X_tr_df, X_val_df = X_cv_reset.iloc[tr_idx], X_cv_reset.iloc[val_idx]
        y_tr,    y_val    = y_cv_reset.iloc[tr_idx], y_cv_reset.iloc[val_idx]

        best_f1, best_combo_fold, best_probs_val = -1, None, None

        for combo in grid:

            # ---- LightGBM ----
            if mt_name == 'lightgbm':
                params = {**LGBM_FIXED, **combo}
                clf = lgb.LGBMClassifier(**params)
                clf.fit(X_tr_df, y_tr, categorical_feature=cat_features)
                val_prob = clf.predict_proba(X_val_df)[:, 1]

            # ---- XGBoost ----
            elif mt_name == 'xgboost':
                X_tr_enc  = X_cv_enc_full.iloc[tr_idx]
                X_val_enc = X_cv_enc_full.iloc[val_idx]
                scale_pos = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
                params = {**XGB_FIXED, **combo, 'scale_pos_weight': scale_pos}
                clf = xgb_lib.XGBClassifier(**params)
                clf.fit(X_tr_enc, y_tr)
                val_prob = clf.predict_proba(X_val_enc)[:, 1]

            # ---- DNN ----
            elif mt_name == 'dnn':
                # Tensor fold'larini X_cv_t_full uzerinden al
                X_tr_t  = X_cv_t_full[tr_idx]
                X_val_t = X_cv_t_full[val_idx]
                y_tr_t  = y_cv_t_full[tr_idx]
                y_val_t = y_cv_t_full[val_idx]

                model_tmp = DeepMLP(
                    input_dim_full, combo['hidden_dim'],
                    combo['n_layers'], combo['dropout']
                )
                criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_full)
                optimizer = optim.AdamW(
                    model_tmp.parameters(), lr=combo['lr'],
                    weight_decay=DNN_FIXED_FAST['weight_decay']
                )
                scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=DNN_FIXED_FAST['max_epochs']
                )
                bs = min(DNN_FIXED_FAST['batch_size'], len(X_tr_t) - 1)
                n  = len(X_tr_t)
                best_vf1, patience = 0, 0

                model_tmp.train()
                for ep in range(DNN_FIXED_FAST['max_epochs']):
                    perm = torch.randperm(n)
                    for i in range(0, n, bs):
                        idx = perm[i:i+bs]
                        if len(idx) < 2: continue
                        optimizer.zero_grad()
                        criterion(model_tmp(X_tr_t[idx]), y_tr_t[idx]).backward()
                        optimizer.step()
                    scheduler.step()
                    model_tmp.eval()
                    with torch.no_grad():
                        vp = torch.sigmoid(model_tmp(X_val_t)).numpy()
                        vf1 = f1_score(y_val_t.numpy(), (vp >= 0.5).astype(int))
                    if vf1 > best_vf1:
                        best_vf1, patience = vf1, 0
                    else:
                        patience += 1
                        if patience >= DNN_FIXED_FAST['patience']: break
                    model_tmp.train()

                model_tmp.eval()
                with torch.no_grad():
                    val_prob = torch.sigmoid(model_tmp(X_val_t)).numpy().ravel()

            # ---- SVM ----
            elif mt_name == 'svm':
                X_tr_enc  = X_cv_enc_full.iloc[tr_idx]
                X_val_enc = X_cv_enc_full.iloc[val_idx]
                scaler_tmp = SKStandardScaler()
                X_tr_sc  = scaler_tmp.fit_transform(X_tr_enc)
                X_val_sc = scaler_tmp.transform(X_val_enc)
                params = {**SVM_FIXED, **combo}
                clf = SVC(**params)
                clf.fit(X_tr_sc, y_tr)
                val_prob = clf.predict_proba(X_val_sc)[:, 1]

            _, f1 = optimize_threshold(y_val, val_prob)
            if f1 > best_f1:
                best_f1         = f1
                best_combo_fold = combo
                best_probs_val  = val_prob

        oof_probs[val_idx, 1] = best_probs_val
        oof_probs[val_idx, 0] = 1 - best_probs_val
        combo_key = str(best_combo_fold)
        combo_votes[combo_key] = combo_votes.get(combo_key, 0) + 1
        print(f'  Fold {fold_idx+1}/{N_FOLDS}: best_combo={best_combo_fold}  F1={best_f1:.4f}')

    # En cok oy alan combo -> final model
    best_combo_final = eval(max(combo_votes, key=combo_votes.get))
    print(f'\n  Final combo (majority vote): {best_combo_final}')

    # ---- Final model: tum X_cv ----
    if mt_name == 'lightgbm':
        final_params = {**LGBM_FIXED, **best_combo_final}
        final_model  = lgb.LGBMClassifier(**final_params)
        final_model.fit(X_cv_reset, y_cv_reset, categorical_feature=cat_features)
        ho_proba_2col = final_model.predict_proba(X_holdout)

    elif mt_name == 'xgboost':
        scale_pos = (y_cv_reset == 0).sum() / max((y_cv_reset == 1).sum(), 1)
        final_params = {**XGB_FIXED, **best_combo_final, 'scale_pos_weight': scale_pos}
        final_model  = xgb_lib.XGBClassifier(**final_params)
        final_model.fit(X_cv_enc_full, y_cv_reset)
        ho_proba_2col = final_model.predict_proba(X_ho_enc_full)

    elif mt_name == 'dnn':
        final_model = DeepMLP(
            input_dim_full, best_combo_final['hidden_dim'],
            best_combo_final['n_layers'], best_combo_final['dropout']
        )
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_full)
        optimizer = optim.AdamW(
            final_model.parameters(), lr=best_combo_final['lr'],
            weight_decay=DNN_FIXED_FAST['weight_decay']
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=DNN_FIXED_FAST['max_epochs']
        )
        bs = min(DNN_FIXED_FAST['batch_size'], len(X_cv_t_full) - 1)
        n  = len(X_cv_t_full)
        best_vf1, best_state, patience = 0, None, 0

        final_model.train()
        for ep in range(DNN_FIXED_FAST['max_epochs']):
            perm = torch.randperm(n)
            for i in range(0, n, bs):
                idx = perm[i:i+bs]
                if len(idx) < 2: continue
                optimizer.zero_grad()
                criterion(final_model(X_cv_t_full[idx]), y_cv_t_full[idx]).backward()
                optimizer.step()
            scheduler.step()
            final_model.eval()
            with torch.no_grad():
                vp = torch.sigmoid(final_model(X_ho_t_full)).numpy()
                vf1 = f1_score(y_holdout.values, (vp >= 0.5).astype(int))
            if vf1 > best_vf1:
                best_vf1  = vf1
                best_state = deepcopy(final_model.state_dict())
                patience  = 0
            else:
                patience += 1
                if patience >= DNN_FIXED_FAST['patience'] + 5: break
            final_model.train()
        if best_state is not None:
            final_model.load_state_dict(best_state)
        final_model.eval()
        with torch.no_grad():
            p_path = torch.sigmoid(final_model(X_ho_t_full)).numpy().ravel()
        ho_proba_2col = np.column_stack([1 - p_path, p_path])

    elif mt_name == 'svm':
        svm_scaler_final = SKStandardScaler()
        X_cv_sc = svm_scaler_final.fit_transform(X_cv_enc_full)
        X_ho_sc = svm_scaler_final.transform(X_ho_enc_full)
        final_params = {**SVM_FIXED, **best_combo_final}
        final_model  = SVC(**final_params)
        final_model.fit(X_cv_sc, y_cv_reset)
        ho_proba_2col = final_model.predict_proba(X_ho_sc)

    ho_prob_1d  = ho_proba_2col[:, 1]
    best_thr, _ = optimize_threshold(y_holdout, ho_prob_1d)
    ho_pred     = (ho_prob_1d >= best_thr).astype(int)
    m_ho        = compute_all_metrics(y_holdout, ho_pred, ho_prob_1d)

    elapsed = time.time() - t0
    print(f'  Hold-out -> F1={m_ho["f1"]:.4f}  AUC={m_ho["auc_roc"]:.4f}  ({elapsed:.1f}s)')

    base_results[mt_name] = {
        'final_model':   final_model,
        'thr':           best_thr,
        'oof_probs':     oof_probs,
        'holdout_probs': ho_proba_2col,
    }
    base_train_log.append({
        'model': mt_name, **m_ho,
        'elapsed_sec': round(elapsed, 1),
        'best_params': str(best_combo_final),
    })

print('\n\nTum baz modeller OOF ile egitildi.')



BAZ MODEL (OOF): LIGHTGBM
  Grid: 12 kombinasyon
  Fold 1/5: best_combo={'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.05}  F1=0.9539
  Fold 2/5: best_combo={'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.1}  F1=0.9602
  Fold 3/5: best_combo={'n_estimators': 100, 'num_leaves': 127, 'learning_rate': 0.05}  F1=0.9575
  Fold 4/5: best_combo={'n_estimators': 100, 'num_leaves': 127, 'learning_rate': 0.1}  F1=0.9713
  Fold 5/5: best_combo={'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.1}  F1=0.9380

  Final combo (majority vote): {'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.1}
  Hold-out -> F1=0.9454  AUC=0.9715  (45.9s)

BAZ MODEL (OOF): XGBOOST
  Grid: 12 kombinasyon
  Fold 1/5: best_combo={'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05}  F1=0.9532
  Fold 2/5: best_combo={'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.05}  F1=0.9553
  Fold 3/5: best_combo={'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05} 

In [16]:
# Cell 5: Meta-Feature Matrislerini Olustur
#
# Meta egitim  : OOF skorlarindan (X_cv boyutunda, temiz)
# Meta test    : Final model holdout tahminlerinden
#
# Sutun sirasi: lgbm_neg, lgbm_pos, xgb_neg, xgb_pos,
#               dnn_neg, dnn_pos, svm_neg, svm_pos

def build_meta_matrix(base_results, split):
    """split: 'oof_probs' veya 'holdout_probs'"""
    parts, col_names = [], []
    for mt in BASE_MODELS:
        proba = base_results[mt][split]  # (N, 2)
        parts.append(proba)
        col_names += [f'{mt}_neg', f'{mt}_pos']
    return np.hstack(parts), col_names

X_meta_feat, meta_col_names = build_meta_matrix(base_results, 'oof_probs')
X_ho_feat,   _              = build_meta_matrix(base_results, 'holdout_probs')

# Meta egitim etiketi = y_cv (OOF ile eslesmis)
y_meta_train = y_cv_reset

print(f'Meta egitim feature matrisi : {X_meta_feat.shape}  -> {meta_col_names}')
print(f'Hold-out feature matrisi    : {X_ho_feat.shape}')
print(f'Meta egitim etiket dagilimi : {pd.Series(y_meta_train.values).value_counts().to_dict()}')
print(f'Hold-out etiket dagilimi    : {pd.Series(y_holdout.values).value_counts().to_dict()}')


Meta egitim feature matrisi : (2524, 8)  -> ['lightgbm_neg', 'lightgbm_pos', 'xgboost_neg', 'xgboost_pos', 'dnn_neg', 'dnn_pos', 'svm_neg', 'svm_pos']
Hold-out feature matrisi    : (632, 8)
Meta egitim etiket dagilimi : {1.0: 1745, 0.0: 779}
Hold-out etiket dagilimi    : {1.0: 437, 0.0: 195}


In [17]:
# Cell 6: Meta-Model 1 — LightGBM Stacking
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

print('=== META-MODEL: LightGBM ===')

meta_lgb_grid = [
    {'n_estimators': ne, 'num_leaves': nl, 'learning_rate': lr}
    for ne in [50, 100, 200]
    for nl in [15, 31]
    for lr in [0.05, 0.1]
]
print(f'LightGBM meta-grid: {len(meta_lgb_grid)} kombinasyon')

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
best_lgb_f1, best_lgb_combo, best_lgb_model = -1, None, None

for combo in meta_lgb_grid:
    fold_f1s = []
    for tr_idx, val_idx in cv.split(X_meta_feat, y_meta_train):
        clf = lgb.LGBMClassifier(
            random_state=SEED, verbose=-1, **combo
        )
        clf.fit(X_meta_feat[tr_idx], y_meta_train.iloc[tr_idx])
        val_prob = clf.predict_proba(X_meta_feat[val_idx])[:, 1]
        thr, f1 = optimize_threshold(y_meta_train.iloc[val_idx], val_prob)
        fold_f1s.append(f1)
    mean_f1 = np.mean(fold_f1s)
    if mean_f1 > best_lgb_f1:
        best_lgb_f1, best_lgb_combo = mean_f1, combo

print(f'En iyi combo: {best_lgb_combo}  -> CV F1={best_lgb_f1:.4f}')

# Tum meta egitim verisi uzerinde final model
best_lgb_model = lgb.LGBMClassifier(
    random_state=SEED, verbose=-1, **best_lgb_combo
)
best_lgb_model.fit(X_meta_feat, y_meta_train)

# Hold-out degerlendirme
ho_prob_lgb = best_lgb_model.predict_proba(X_ho_feat)[:, 1]
best_lgb_thr, _ = optimize_threshold(y_holdout, ho_prob_lgb)
ho_pred_lgb = (ho_prob_lgb >= best_lgb_thr).astype(int)
metrics_lgb = compute_all_metrics(y_holdout, ho_pred_lgb, ho_prob_lgb)

print(f'\nHold-out (LightGBM meta):')
print(f'  F1={metrics_lgb["f1"]:.4f}  AUC={metrics_lgb["auc_roc"]:.4f}  '
      f'P={metrics_lgb["precision"]:.4f}  R={metrics_lgb["recall"]:.4f}  '
      f'MCC={metrics_lgb["mcc"]:.4f}')

=== META-MODEL: LightGBM ===
LightGBM meta-grid: 12 kombinasyon
En iyi combo: {'n_estimators': 50, 'num_leaves': 15, 'learning_rate': 0.05}  -> CV F1=0.9528

Hold-out (LightGBM meta):
  F1=0.9444  AUC=0.9620  P=0.9179  R=0.9725  MCC=0.8116


In [18]:
# Cell 7: Meta-Model 2 — NN Stacking
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold

print('=== META-MODEL: NN ===')

META_INPUT_DIM = X_meta_feat.shape[1]  # 8


class MetaMLP(nn.Module):
    """Kucuk 2-katmanli MLP (8 -> 32 -> 16 -> 1)"""
    def __init__(self, input_dim=8, hidden1=32, hidden2=16, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_meta_nn(X_tr, y_tr, X_val, y_val, dropout=0.2, lr=1e-3,
                  hidden1=32, hidden2=16, max_epochs=100, patience=15):
    pos_weight = torch.tensor(
        [(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)], dtype=torch.float
    )
    model = MetaMLP(X_tr.shape[1], hidden1, hidden2, dropout)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    X_tr_t  = torch.tensor(X_tr,  dtype=torch.float)
    y_tr_t  = torch.tensor(y_tr,  dtype=torch.float)
    X_val_t = torch.tensor(X_val, dtype=torch.float)
    y_val_t = torch.tensor(y_val, dtype=torch.float)

    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=64, shuffle=True)

    best_val_loss, best_state, no_improve = float('inf'), None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss, best_state, no_improve = val_loss, {
                k: v.clone() for k, v in model.state_dict().items()
            }, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    return model


# Normalize meta features
meta_scaler = StandardScaler()
X_meta_scaled = meta_scaler.fit_transform(X_meta_feat)
X_ho_scaled   = meta_scaler.transform(X_ho_feat)

y_meta_arr = y_meta_train.values.astype(float)
y_ho_arr   = y_holdout.values.astype(float)

meta_nn_grid = [
    {'dropout': d, 'lr': lr}
    for d  in [0.1, 0.2, 0.3]
    for lr in [1e-3, 3e-4]
]
print(f'NN meta-grid: {len(meta_nn_grid)} kombinasyon')

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
best_nn_f1, best_nn_combo = -1, None

for combo in meta_nn_grid:
    fold_f1s = []
    for tr_idx, val_idx in cv.split(X_meta_scaled, y_meta_arr):
        m = train_meta_nn(
            X_meta_scaled[tr_idx], y_meta_arr[tr_idx],
            X_meta_scaled[val_idx], y_meta_arr[val_idx],
            **combo
        )
        m.eval()
        with torch.no_grad():
            val_prob = torch.sigmoid(
                m(torch.tensor(X_meta_scaled[val_idx], dtype=torch.float))
            ).numpy()
        _, f1 = optimize_threshold(y_meta_arr[val_idx], val_prob)
        fold_f1s.append(f1)
    mean_f1 = np.mean(fold_f1s)
    print(f'  combo={combo}  CV F1={mean_f1:.4f}')
    if mean_f1 > best_nn_f1:
        best_nn_f1, best_nn_combo = mean_f1, combo

print(f'En iyi NN combo: {best_nn_combo}  -> CV F1={best_nn_f1:.4f}')

# Final NN egitimi (tum meta verisi uzerinde, fold ayrimi yok)
val_split = int(0.1 * len(X_meta_scaled))
best_nn_model = train_meta_nn(
    X_meta_scaled[val_split:], y_meta_arr[val_split:],
    X_meta_scaled[:val_split],  y_meta_arr[:val_split],
    **best_nn_combo
)

best_nn_model.eval()
with torch.no_grad():
    ho_prob_nn = torch.sigmoid(
        best_nn_model(torch.tensor(X_ho_scaled, dtype=torch.float))
    ).numpy()

best_nn_thr, _ = optimize_threshold(y_holdout, ho_prob_nn)
ho_pred_nn = (ho_prob_nn >= best_nn_thr).astype(int)
metrics_nn = compute_all_metrics(y_holdout, ho_pred_nn, ho_prob_nn)

print(f'\nHold-out (NN meta):')
print(f'  F1={metrics_nn["f1"]:.4f}  AUC={metrics_nn["auc_roc"]:.4f}  '
      f'P={metrics_nn["precision"]:.4f}  R={metrics_nn["recall"]:.4f}  '
      f'MCC={metrics_nn["mcc"]:.4f}')

=== META-MODEL: NN ===
NN meta-grid: 6 kombinasyon
  combo={'dropout': 0.1, 'lr': 0.001}  CV F1=0.9567
  combo={'dropout': 0.1, 'lr': 0.0003}  CV F1=0.9570
  combo={'dropout': 0.2, 'lr': 0.001}  CV F1=0.9569
  combo={'dropout': 0.2, 'lr': 0.0003}  CV F1=0.9564
  combo={'dropout': 0.3, 'lr': 0.001}  CV F1=0.9572
  combo={'dropout': 0.3, 'lr': 0.0003}  CV F1=0.9570
En iyi NN combo: {'dropout': 0.3, 'lr': 0.001}  -> CV F1=0.9572

Hold-out (NN meta):
  F1=0.9505  AUC=0.9664  P=0.9357  R=0.9657  MCC=0.8347


In [19]:
# Cell 7A: Diversity-Aware Meta-Feature Matrisi
#
# Standart 1x8 vektore (her modelin neg+pos skoru) ek olarak
# modeller-arasi cesitlilik (diversity) sinyalleri eklenir:
#
#   disagreement : 4 pos-skorun standart sapmasi          (1 sutun)
#   confidence   : 4 pos-skorun ortalamasi                (1 sutun)
#   entropy      : binary entropi (confidence uzerinden)  (1 sutun)
#   max_min_gap  : max(pos) - min(pos)                    (1 sutun)
#   pairwise_diffs: her model ciftinin |fark|             (6 sutun, C(4,2))
#
# Toplam: 8 (orijinal) + 11 (diversity) = 19 meta-feature sutun

from itertools import combinations

def add_diversity_features(X_meta, col_names):
    pos_idx    = [i for i, c in enumerate(col_names) if c.endswith('_pos')]
    pos_scores = X_meta[:, pos_idx]                                    # (N, 4)
    model_tags = [col_names[i].replace('_pos', '') for i in pos_idx]

    disagreement = pos_scores.std(axis=1, keepdims=True)              # (N,1)
    confidence   = pos_scores.mean(axis=1, keepdims=True)             # (N,1)
    eps = 1e-9
    p   = np.clip(confidence, eps, 1 - eps)
    entropy      = -(p * np.log(p) + (1-p) * np.log(1-p))            # (N,1)
    max_min_gap  = (pos_scores.max(axis=1) - pos_scores.min(axis=1)).reshape(-1, 1)  # (N,1)

    pairwise_parts, pairwise_names = [], []
    for (i, m1), (j, m2) in combinations(enumerate(model_tags), 2):
        diff = np.abs(pos_scores[:, i] - pos_scores[:, j]).reshape(-1, 1)
        pairwise_parts.append(diff)
        pairwise_names.append(f'diff_{m1}_{m2}')

    diversity_block = np.hstack([disagreement, confidence, entropy,
                                  max_min_gap] + pairwise_parts)
    diversity_names = ['disagreement', 'confidence', 'entropy', 'max_min_gap'] + pairwise_names

    X_ext      = np.hstack([X_meta, diversity_block])
    names_ext  = col_names + diversity_names
    return X_ext, names_ext

X_meta_div, meta_div_col_names = add_diversity_features(X_meta_feat, meta_col_names)
X_ho_div,   _                  = add_diversity_features(X_ho_feat,   meta_col_names)

print(f'Orijinal meta-feature     : {X_meta_feat.shape}  ({len(meta_col_names)} sutun)')
print(f'Diversity-aware meta-feat : {X_meta_div.shape}  ({len(meta_div_col_names)} sutun)')
print(f'Yeni sutunlar             : {meta_div_col_names[8:]}')

div_df = pd.DataFrame(X_meta_div[:, 8:], columns=meta_div_col_names[8:])
print('\nDiversity feature istatistikleri:')
display(div_df.describe().round(4))


Orijinal meta-feature     : (2524, 8)  (8 sutun)
Diversity-aware meta-feat : (2524, 18)  (18 sutun)
Yeni sutunlar             : ['disagreement', 'confidence', 'entropy', 'max_min_gap', 'diff_lightgbm_xgboost', 'diff_lightgbm_dnn', 'diff_lightgbm_svm', 'diff_xgboost_dnn', 'diff_xgboost_svm', 'diff_dnn_svm']

Diversity feature istatistikleri:


,disagreement,confidence,entropy,max_min_gap,diff_lightgbm_xgboost,diff_lightgbm_dnn,diff_lightgbm_svm,diff_xgboost_dnn,diff_xgboost_svm,diff_dnn_svm
count,2524.0000,2524.0000,2524.0000,2524.0000,2524.0000,2524.0000,2524.0000,2524.0000,2524.0000,2524.0000
mean,0.1325,0.6675,0.3343,0.3205,0.0666,0.2314,0.2232,0.2319,0.2004,0.1489
std,0.1251,0.3442,0.2187,0.2947,0.0984,0.3091,0.2335,0.2809,0.2193,0.1436
min,0.0021,0.0026,0.0180,0.0052,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0294,0.4022,0.1340,0.0719,0.0119,0.0059,0.0439,0.0224,0.0317,0.0369
50%,0.0875,0.8372,0.2859,0.2136,0.0259,0.0644,0.1267,0.0879,0.1110,0.0983
75%,0.2047,0.9601,0.5456,0.5091,0.0779,0.3607,0.3390,0.3614,0.3001,0.2200
max,0.4804,0.9969,0.6931,1.0000,0.7091,1.0000,0.9786,0.9868,0.9230,0.7052


In [20]:
# Cell 7B: Diversity-Aware Meta-Modeller (LightGBM + NN)
#
# Genis meta-feature (19 sutun) uzerinde iki meta-model egitilir:
#   diversity_lgbm : LightGBM meta-model (19 input)
#   diversity_nn   : NN meta-model        (19 input, hidden1=64, hidden2=32)
# Her ikisi de 3-fold CV ile secilir; hold-out uzerinde degerlendirilir.

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

META_DIV_INPUT_DIM = X_meta_div.shape[1]  # 19

# ---- 7B.1: Diversity LightGBM ----
print('=== DIVERSITY-AWARE META-MODEL: LightGBM ===')

div_lgb_grid = [
    {'n_estimators': ne, 'num_leaves': nl, 'learning_rate': lr}
    for ne in [50, 100, 200]
    for nl in [15, 31]
    for lr in [0.05, 0.1]
]

cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
best_dlgb_f1, best_dlgb_combo = -1, None

for combo in div_lgb_grid:
    fold_f1s = []
    for tr_idx, val_idx in cv3.split(X_meta_div, y_meta_train):
        clf = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **combo)
        clf.fit(X_meta_div[tr_idx], y_meta_train.iloc[tr_idx])
        val_prob = clf.predict_proba(X_meta_div[val_idx])[:, 1]
        _, f1 = optimize_threshold(y_meta_train.iloc[val_idx], val_prob)
        fold_f1s.append(f1)
    mean_f1 = np.mean(fold_f1s)
    if mean_f1 > best_dlgb_f1:
        best_dlgb_f1, best_dlgb_combo = mean_f1, combo

print(f'En iyi combo: {best_dlgb_combo}  -> CV F1={best_dlgb_f1:.4f}')

best_dlgb_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **best_dlgb_combo)
best_dlgb_model.fit(X_meta_div, y_meta_train)

ho_prob_dlgb = best_dlgb_model.predict_proba(X_ho_div)[:, 1]
best_dlgb_thr, _ = optimize_threshold(y_holdout, ho_prob_dlgb)
ho_pred_dlgb = (ho_prob_dlgb >= best_dlgb_thr).astype(int)
metrics_dlgb = compute_all_metrics(y_holdout, ho_pred_dlgb, ho_prob_dlgb)

print(f'Hold-out (Diversity LightGBM):')
print(f'  F1={metrics_dlgb["f1"]:.4f}  AUC={metrics_dlgb["auc_roc"]:.4f}  '
      f'P={metrics_dlgb["precision"]:.4f}  R={metrics_dlgb["recall"]:.4f}  '
      f'MCC={metrics_dlgb["mcc"]:.4f}')

# ---- 7B.2: Diversity NN ----
print('\n=== DIVERSITY-AWARE META-MODEL: NN ===')

meta_div_scaler = StandardScaler()
X_meta_div_scaled = meta_div_scaler.fit_transform(X_meta_div)
X_ho_div_scaled   = meta_div_scaler.transform(X_ho_div)

y_meta_arr = y_meta_train.values.astype(float)

meta_div_nn_grid = [
    {'dropout': d, 'lr': lr}
    for d  in [0.1, 0.2, 0.3]
    for lr in [1e-3, 3e-4]
]
print(f'NN meta-grid: {len(meta_div_nn_grid)} kombinasyon')

best_dnn_f1, best_dnn_combo = -1, None

for combo in meta_div_nn_grid:
    fold_f1s = []
    for tr_idx, val_idx in cv3.split(X_meta_div_scaled, y_meta_arr):
        m = train_meta_nn(
            X_meta_div_scaled[tr_idx], y_meta_arr[tr_idx],
            X_meta_div_scaled[val_idx], y_meta_arr[val_idx],
            **combo, hidden1=64, hidden2=32
        )
        m.eval()
        with torch.no_grad():
            val_prob = torch.sigmoid(
                m(torch.tensor(X_meta_div_scaled[val_idx], dtype=torch.float))
            ).numpy()
        _, f1 = optimize_threshold(y_meta_arr[val_idx], val_prob)
        fold_f1s.append(f1)
    mean_f1 = np.mean(fold_f1s)
    print(f'  combo={combo}  CV F1={mean_f1:.4f}')
    if mean_f1 > best_dnn_f1:
        best_dnn_f1, best_dnn_combo = mean_f1, combo

print(f'En iyi Diversity NN combo: {best_dnn_combo}  -> CV F1={best_dnn_f1:.4f}')

val_split = int(0.1 * len(X_meta_div_scaled))
best_dnn_model = train_meta_nn(
    X_meta_div_scaled[val_split:], y_meta_arr[val_split:],
    X_meta_div_scaled[:val_split],  y_meta_arr[:val_split],
    **best_dnn_combo, hidden1=64, hidden2=32
)

best_dnn_model.eval()
with torch.no_grad():
    ho_prob_dnn = torch.sigmoid(
        best_dnn_model(torch.tensor(X_ho_div_scaled, dtype=torch.float))
    ).numpy()

best_dnn_thr, _ = optimize_threshold(y_holdout, ho_prob_dnn)
ho_pred_dnn = (ho_prob_dnn >= best_dnn_thr).astype(int)
metrics_dnn = compute_all_metrics(y_holdout, ho_pred_dnn, ho_prob_dnn)

print(f'\nHold-out (Diversity NN):')
print(f'  F1={metrics_dnn["f1"]:.4f}  AUC={metrics_dnn["auc_roc"]:.4f}  '
      f'P={metrics_dnn["precision"]:.4f}  R={metrics_dnn["recall"]:.4f}  '
      f'MCC={metrics_dnn["mcc"]:.4f}')

# ---- Feature importance (Diversity LightGBM) ----
fi_div = best_dlgb_model.feature_importances_
fi_div_df = pd.DataFrame({'feature': meta_div_col_names, 'importance': fi_div}).sort_values(
    'importance', ascending=False
)
print('\nTop-10 Diversity LightGBM Feature Importance:')
display(fi_div_df.head(10))


=== DIVERSITY-AWARE META-MODEL: LightGBM ===
En iyi combo: {'n_estimators': 50, 'num_leaves': 31, 'learning_rate': 0.05}  -> CV F1=0.9530
Hold-out (Diversity LightGBM):
  F1=0.9445  AUC=0.9606  P=0.9350  R=0.9542  MCC=0.8164

=== DIVERSITY-AWARE META-MODEL: NN ===
NN meta-grid: 6 kombinasyon
  combo={'dropout': 0.1, 'lr': 0.001}  CV F1=0.9565
  combo={'dropout': 0.1, 'lr': 0.0003}  CV F1=0.9571
  combo={'dropout': 0.2, 'lr': 0.001}  CV F1=0.9574
  combo={'dropout': 0.2, 'lr': 0.0003}  CV F1=0.9568
  combo={'dropout': 0.3, 'lr': 0.001}  CV F1=0.9576
  combo={'dropout': 0.3, 'lr': 0.0003}  CV F1=0.9572
En iyi Diversity NN combo: {'dropout': 0.3, 'lr': 0.001}  -> CV F1=0.9576

Hold-out (Diversity NN):
  F1=0.9495  AUC=0.9661  P=0.9517  R=0.9474  MCC=0.8373

Top-10 Diversity LightGBM Feature Importance:


,feature,importance
15,diff_xgboost_dnn,142
12,diff_lightgbm_xgboost,126
3,xgboost_pos,121
9,confidence,116
17,diff_dnn_svm,105
16,diff_xgboost_svm,99
6,svm_neg,96
2,xgboost_neg,89
10,entropy,88
14,diff_lightgbm_svm,86


In [21]:
# Cell 8: Sonuclari Birlestir & Kaydet
from datetime import datetime

# --- Baz model sonuclari ---
base_df = pd.DataFrame(base_train_log)
base_df['type'] = 'base'
base_df['panel'] = PANEL

# --- Standart meta model sonuclari ---
meta_rows = [
    {'model': 'stacking_lgbm', 'type': 'meta', 'panel': PANEL,
     'threshold': best_lgb_thr, 'best_params': str(best_lgb_combo), **metrics_lgb},
    {'model': 'stacking_nn',   'type': 'meta', 'panel': PANEL,
     'threshold': best_nn_thr,  'best_params': str(best_nn_combo),  **metrics_nn},
]
meta_df = pd.DataFrame(meta_rows)

# --- Diversity-aware meta model sonuclari ---
diversity_rows = [
    {'model': 'diversity_lgbm', 'type': 'diversity', 'panel': PANEL,
     'threshold': best_dlgb_thr, 'best_params': str(best_dlgb_combo), **metrics_dlgb},
    {'model': 'diversity_nn',   'type': 'diversity', 'panel': PANEL,
     'threshold': best_dnn_thr,  'best_params': str(best_dnn_combo),  **metrics_dnn},
]
diversity_df = pd.DataFrame(diversity_rows)

all_df = pd.concat([base_df, meta_df, diversity_df], ignore_index=True)
all_df.to_csv(os.path.join(RESULTS_STACK_DIR, 'stacking_results.csv'), index=False)

print('=== TAM SONUC TABLOSU (Baz + Meta + Diversity) ===')
display(all_df[['type','model','f1','auc_roc','precision','recall','mcc','balanced_accuracy']])


=== TAM SONUC TABLOSU (Baz + Meta + Diversity) ===


,type,model,f1,auc_roc,precision,recall,mcc,balanced_accuracy
0,base,lightgbm,0.945373,0.971496,0.921739,0.970252,0.815336,0.892818
1,base,xgboost,0.945129,0.968233,0.925439,0.965675,0.815397,0.895658
2,base,dnn,0.884995,0.890395,0.848739,0.924485,0.594818,0.777627
3,base,svm,0.895490,0.896028,0.862288,0.931350,0.635244,0.799008
4,meta,stacking_lgbm,0.944444,0.961979,0.917927,0.972540,0.811556,0.888834
5,meta,stacking_nn,0.950450,0.966367,0.935698,0.965675,0.834699,0.908479
6,diversity,diversity_lgbm,0.944507,0.960629,0.934978,0.954233,0.816409,0.902758
7,diversity,diversity_nn,0.949541,0.966133,0.951724,0.947368,0.837321,0.919838


In [22]:
# Cell 9: Gorsellestirme
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve

fig_paths = []

# 1. F1 karsilastirma bar chart (baz / meta / diversity)
fig, ax = plt.subplots(figsize=(12, 5))
color_map = {'base': '#4CAF50', 'meta': '#2196F3', 'diversity': '#FF9800'}
colors = [color_map[r['type']] for _, r in all_df.iterrows()]
bars = ax.bar(all_df['model'], all_df['f1'], color=colors)
ax.bar_label(bars, fmt='%.4f', fontsize=8)
ax.set_ylim(0, 1.10)
ax.set_title('Hold-Out F1 - Baz (yesil) / Meta (mavi) / Diversity (turuncu)', fontsize=13)
ax.set_ylabel('F1 Score')
ax.set_xlabel('Model')
ax.axhline(all_df.loc[all_df['type']=='base','f1'].max(),
           color='#4CAF50', linestyle='--', linewidth=1, label='En iyi baz model')
ax.legend()
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'f1_comparison.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 2. Meta-feature korelasyon isisi haritasi (orijinal 8 sutun)
meta_corr_df = pd.DataFrame(X_meta_feat, columns=meta_col_names)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(meta_corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, center=0)
ax.set_title('Meta-Feature Korelasyon Matrisi (1x8 Vektor)', fontsize=12)
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'meta_feature_correlation.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 3. Diversity feature korelasyon isisi haritasi (11 yeni sutun)
div_feat_df = pd.DataFrame(X_meta_div[:, 8:], columns=meta_div_col_names[8:])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(div_feat_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, center=0)
ax.set_title('Diversity Feature Korelasyon Matrisi (11 yeni sutun)', fontsize=12)
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'diversity_feature_correlation.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 4. ROC egrileri (tum modeller)
best_base_name = all_df.loc[all_df['type']=='base','f1'].idxmax()
best_base_name = all_df.loc[best_base_name, 'model']

fig, ax = plt.subplots(figsize=(8, 6))
for label, probs in [
    ('Stacking LightGBM',   ho_prob_lgb),
    ('Stacking NN',         ho_prob_nn.ravel()),
    ('Diversity LightGBM',  ho_prob_dlgb),
    ('Diversity NN',        ho_prob_dnn.ravel()),
    (f'Baz {best_base_name}', base_results[best_base_name]['holdout_probs'][:, 1]),
]:
    fpr, tpr, _ = roc_curve(y_holdout, probs)
    auc = compute_all_metrics(y_holdout, (probs >= 0.5).astype(int), probs)['auc_roc']
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.4f})')
ax.plot([0,1],[0,1],'k--')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Egrileri - Tum Modeller')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'roc_curves.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 5. Precision-Recall egrileri
fig, ax = plt.subplots(figsize=(8, 6))
for label, probs in [
    ('Stacking LightGBM',   ho_prob_lgb),
    ('Stacking NN',         ho_prob_nn.ravel()),
    ('Diversity LightGBM',  ho_prob_dlgb),
    ('Diversity NN',        ho_prob_dnn.ravel()),
    (f'Baz {best_base_name}', base_results[best_base_name]['holdout_probs'][:, 1]),
]:
    prec, rec, _ = precision_recall_curve(y_holdout, probs)
    auc_pr = compute_all_metrics(y_holdout, (probs >= 0.5).astype(int), probs)['auc_pr']
    ax.plot(rec, prec, label=f'{label} (AUC-PR={auc_pr:.4f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Egrileri')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'pr_curves.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 6. Confusion matrisler (meta + diversity)
for label, pred, prob in [
    ('Stacking LightGBM',  ho_pred_lgb,  ho_prob_lgb),
    ('Stacking NN',        ho_pred_nn,   ho_prob_nn.ravel()),
    ('Diversity LightGBM', ho_pred_dlgb, ho_prob_dlgb),
    ('Diversity NN',       ho_pred_dnn,  ho_prob_dnn.ravel()),
]:
    cm = confusion_matrix(y_holdout, pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(
        ax=ax, cmap='Blues'
    )
    ax.set_title(f'Confusion Matrix - {label}', fontsize=12)
    plt.tight_layout()
    fname = label.lower().replace(' ', '_')
    p = os.path.join(RESULTS_STACK_DIR, f'{fname}_confusion_matrix.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 7. Meta-feature dagilim boxplot
fig, ax = plt.subplots(figsize=(10, 5))
meta_corr_df['label'] = y_meta_train.values
melt_df = meta_corr_df.melt(id_vars='label', value_vars=meta_col_names)
sns.boxplot(data=melt_df, x='variable', y='value', hue='label',
            palette={0: '#4CAF50', 1: '#F44336'}, ax=ax)
ax.set_title('Meta-Feature Dagilimi (Benign=0 vs Pathogenic=1)', fontsize=12)
ax.set_xlabel('Meta Feature'); ax.set_ylabel('Olasilik Skoru')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'meta_feature_distribution.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 8. Diversity LightGBM feature importance (19 feature)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(fi_div_df['feature'], fi_div_df['importance'], color='#FF9800')
ax.invert_yaxis()
ax.set_title('Diversity LightGBM - Feature Importance (19 Feature)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'diversity_lgbm_feature_importance.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

# 9. Standart LightGBM meta model feature importance
fi = best_lgb_model.feature_importances_
fi_df = pd.DataFrame({'feature': meta_col_names, 'importance': fi}).sort_values(
    'importance', ascending=False
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(fi_df['feature'], fi_df['importance'], color='#2196F3')
ax.invert_yaxis()
ax.set_title('Standart Meta LightGBM - Feature Importance (8 Feature)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
p = os.path.join(RESULTS_STACK_DIR, 'meta_lgbm_feature_importance.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.show()

print(f'Toplam {len(fig_paths)} grafik kaydedildi.')


Toplam 12 grafik kaydedildi.


In [23]:
# Cell 10: PDF Rapor
from fpdf import FPDF
from datetime import datetime


class StackingReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 8, 'Teknofest - Stacking Model Raporu (General Panel, No In-Silico)',
                  align='C', new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}/{{nb}}', align='C')


pdf = StackingReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Baslik sayfasi
pdf.add_page()
pdf.set_font('Helvetica', 'B', 20)
pdf.ln(40)
pdf.cell(0, 15, 'Stacking Ensemble Model Raporu', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 13)
pdf.cell(0, 10, 'General Panel - In-Silico Skorlar OLMADAN (v3_no_insil)',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(8)
pdf.set_font('Helvetica', '', 11)
pdf.cell(0, 7, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 7, f'Baz Modeller: {", ".join(BASE_MODELS).upper()}',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 7, f'OOF Fold Sayisi: {N_FOLDS}  |  Meta-feature: 1x8 vektor (4 x [p_benign, p_patho])',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 7, 'Meta Modeller: LightGBM, NN  |  Diversity Meta: LightGBM(19), NN(19)',
         align='C', new_x='LMARGIN', new_y='NEXT')

# Veri ozeti
pdf.add_page()
pdf.set_font('Helvetica', 'B', 14)
pdf.cell(0, 10, '1. Veri & Bolunme Ozeti', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
rows_info = [
    f'Toplam veri: {len(df_raw)} satir',
    f'General panel: {len(y)} satir  (pos={int(y.sum())}, neg={int((y==0).sum())})',
    f'CV seti (OOF baz egitim + meta egitim): {len(y_cv)} satir',
    f'Hold-out (test): {len(y_holdout)} satir',
    f'Feature sayisi (FE sonrasi): {X.shape[1]}',
    f'Standart meta feature: {META_INPUT_DIM}  (4 model x 2 skor)',
    f'Diversity meta feature: {META_DIV_INPUT_DIM}  (8 orijinal + 11 diversity)',
    f'OOF fold sayisi: {N_FOLDS}',
]
for r in rows_info:
    pdf.cell(0, 7, r, new_x='LMARGIN', new_y='NEXT')

col_w = [35, 20, 20, 20, 20, 20, 20]
hdrs  = ['Model', 'F1', 'AUC-ROC', 'Prec', 'Recall', 'MCC', 'Bal.Acc']

def write_result_table(pdf, df, title):
    pdf.ln(3)
    pdf.set_font('Helvetica', 'B', 13)
    pdf.cell(0, 10, title, new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', 'B', 9)
    for w, h in zip(col_w, hdrs):
        pdf.cell(w, 7, h, border=1, align='C')
    pdf.ln()
    pdf.set_font('Helvetica', '', 8)
    for _, row in df.iterrows():
        vals = [
            row['model'], f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}",
            f"{row['precision']:.4f}", f"{row['recall']:.4f}",
            f"{row['mcc']:.4f}", f"{row['balanced_accuracy']:.4f}",
        ]
        for w, v in zip(col_w, vals):
            pdf.cell(w, 6, str(v), border=1, align='C')
        pdf.ln()

pdf.add_page()
write_result_table(pdf, base_df,      '2. Baz Model Hold-Out Sonuclari')
write_result_table(pdf, meta_df,      '3. Standart Meta-Model Sonuclari')
write_result_table(pdf, diversity_df, '4. Diversity-Aware Meta-Model Sonuclari')

# Diversity feature aciklamasi
pdf.add_page()
pdf.set_font('Helvetica', 'B', 13)
pdf.cell(0, 10, '5. Diversity Feature Aciklamasi', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 9)
div_desc = [
    'disagreement : 4 baz modelin pos-skor standart sapmasi',
    'confidence   : 4 baz modelin pos-skor ortalamasi (ensemble guven skoru)',
    'entropy      : binary entropi (confidence uzerinden; 0.5 civarinda max belirsizlik)',
    'max_min_gap  : en yuksek ile en dusuk pos-skor farki',
    'diff_lgbm_xgb: LightGBM ile XGBoost pos-skor mutlak farki',
    'diff_lgbm_dnn: LightGBM ile DNN pos-skor mutlak farki',
    'diff_lgbm_svm: LightGBM ile SVM pos-skor mutlak farki',
    'diff_xgb_dnn : XGBoost ile DNN pos-skor mutlak farki',
    'diff_xgb_svm : XGBoost ile SVM pos-skor mutlak farki',
    'diff_dnn_svm : DNN ile SVM pos-skor mutlak farki',
    '',
    'Toplam: 8 (orijinal) + 11 (diversity) = 19 meta-feature',
]
for line in div_desc:
    pdf.cell(0, 6, line, new_x='LMARGIN', new_y='NEXT')

# Grafikler
for fp in fig_paths:
    if os.path.exists(fp):
        pdf.add_page()
        fname = os.path.basename(fp).replace('.png', '').replace('_', ' ').title()
        pdf.set_font('Helvetica', 'B', 12)
        pdf.cell(0, 10, fname, new_x='LMARGIN', new_y='NEXT')
        try:
            pdf.image(fp, x=10, w=190)
        except Exception as e:
            pdf.set_font('Helvetica', '', 9)
            pdf.cell(0, 7, f'Grafik yuklenemedi: {e}', new_x='LMARGIN', new_y='NEXT')

report_path = os.path.join(REPORTS_DIR, 'stacking_general_panel_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')


PDF rapor kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\stacking_general_panel_report.pdf
